<a href="https://colab.research.google.com/github/KP-365/Skinrash-detection/blob/main/BugBites_NoAug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from datasets import load_dataset
import numpy as np

SEED = 1337
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 8
BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS_HEAD = 30
EPOCHS_FINETUNE = 10
LR_HEAD = 1e-3
LR_FINETUNE = 1e-5
DROPOUT_P = 0.3
PATIENCE = 5

class EarlyStopping:
    """Tracks validation LOSS to decide when to stop training."""
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

clean = load_dataset("eceunal/bug-bite-images-hf")
labels = clean["train"].features["label"].names
print("Classes:", labels)
print(f"Clean train size: {len(clean['train'])}")
print(f"Clean validation size: {len(clean['validation'])}")
print(f"Clean test size: {len(clean['test'])}")

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class BugBiteDataset(Dataset):
    def __init__(self, hf_split, transform):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]
        img = ex["image"].convert("RGB")
        img = self.transform(img)
        return img, ex["label"]

train_ds = BugBiteDataset(clean["train"], train_transform)
val_ds = BugBiteDataset(clean["validation"], eval_transform)
test_ds = BugBiteDataset(clean["test"], eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

def build_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for param in model.features.parameters():
        param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=DROPOUT_P),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)

model = build_model()

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, targets in loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, targets in loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == targets).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

criterion = nn.CrossEntropyLoss()

# ============================================================
# Phase 1: head-only training
# ============================================================
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)
early_stopper = EarlyStopping(patience=PATIENCE)

print("\n=== [CLEAN-ONLY MODEL] Phase 1: Head-only training ===")
best_val_acc = 0
best_epoch_phase1 = 0
for epoch in range(EPOCHS_HEAD):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS_HEAD} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch_phase1 = epoch + 1
        torch.save(model.state_dict(), "best_model_clean_phase1.pt")
        print(f"  -> New best (epoch {epoch+1}), checkpoint saved.")

    early_stopper(val_loss)
    if early_stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch+1} (phase 1)")
        break

print(f"\nPhase 1 complete. Best val_acc={best_val_acc:.4f} at epoch {best_epoch_phase1}")

# ============================================================
# Phase 2: fine-tune last block
# ============================================================
print("\n=== [CLEAN-ONLY MODEL] Phase 2: Fine-tuning last block ===")
model.load_state_dict(torch.load("best_model_clean_phase1.pt"))
for param in model.features[-1].parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINETUNE)
early_stopper = EarlyStopping(patience=PATIENCE)

best_val_acc = 0
best_epoch_phase2 = 0
for epoch in range(EPOCHS_FINETUNE):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS_FINETUNE} | train_acc={train_acc:.4f} | val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch_phase2 = epoch + 1
        torch.save(model.state_dict(), "best_model_clean_only_final.pt")
        print(f"  -> New best (epoch {epoch+1}), checkpoint saved.")

    early_stopper(val_loss)
    if early_stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch+1} (phase 2)")
        break

print(f"\nPhase 2 complete. Best val_acc={best_val_acc:.4f} at epoch {best_epoch_phase2}")
print(f"Final saved checkpoint: best_model_clean_only_final.pt (from phase 2, epoch {best_epoch_phase2})")

# ============================================================
# Final test evaluation -- loads the BEST checkpoint, not the last epoch
# ============================================================
model.load_state_dict(torch.load("best_model_clean_only_final.pt"))
test_loss, test_acc = eval_epoch(model, test_loader, criterion)
print(f"\n[CLEAN-ONLY MODEL] Final TEST accuracy: {test_acc:.4f} | test_loss: {test_loss:.4f}")

Classes: ['ants', 'bed_bugs', 'chiggers', 'fleas', 'mosquitos', 'no_bites', 'spiders', 'ticks']
Clean train size: 896
Clean validation size: 106
Clean test size: 53

=== [CLEAN-ONLY MODEL] Phase 1: Head-only training ===
Epoch 1/30 | train_acc=0.2634 | val_acc=0.3868
  -> New best (epoch 1), checkpoint saved.
Epoch 2/30 | train_acc=0.4866 | val_acc=0.4245
  -> New best (epoch 2), checkpoint saved.
Epoch 3/30 | train_acc=0.5190 | val_acc=0.4906
  -> New best (epoch 3), checkpoint saved.
Epoch 4/30 | train_acc=0.5603 | val_acc=0.4906
Epoch 5/30 | train_acc=0.5960 | val_acc=0.4811
Epoch 6/30 | train_acc=0.6172 | val_acc=0.4623
Epoch 7/30 | train_acc=0.6172 | val_acc=0.4906
Epoch 8/30 | train_acc=0.6027 | val_acc=0.4717
Epoch 9/30 | train_acc=0.5971 | val_acc=0.5000
  -> New best (epoch 9), checkpoint saved.
Epoch 10/30 | train_acc=0.6674 | val_acc=0.5094
  -> New best (epoch 10), checkpoint saved.
Epoch 11/30 | train_acc=0.6328 | val_acc=0.5189
  -> New best (epoch 11), checkpoint saved.
